# 🍀 はじめに 🍀
### 🌱このノートブックは、Stable Diffusion WebUI ForgeをGoogle Colabで手軽に起動するために制作しました。
### 🌱Google Driveへの自動保存や、Civitaiからのモデルダウンロードに対応しています。
### Stable Diffusion WebUI Forge（Classic）をGoogle Colabで起動し、AI画像生成を体験できます。
### Google Driveへの自動保存・Civitaiからのモデルダウンロードに対応しています。

---
## 🔑 事前準備：Civitai API キーの設定
モデルをダウンロードするには **Civitai の API キー** をGoogle Colabの「シークレット」に登録してください。

**① API キーの取得**
1. [https://civitai.com/user/account](https://civitai.com/user/account) を開く（要ログイン）
2. ページ下部の **API Keys** セクション → **＋ Add API key** をクリック
3. 名前を入力して **Save** → 表示されたキーをコピー

**② Google Colab シークレットへの登録**
1. 左サイドバーの 🔑 **シークレット**（鍵アイコン）を開く
2. **＋ 新しいシークレットを追加** をクリック
3. 名前に `CIVITAI_KEY`、値にコピーしたAPIキーを貼り付けて保存
4. **ノートブックからのアクセス** をオンにする

> ⚠️ API キーはシークレットで管理し、コードに直接貼り付けないでください。

---
**ノートブック原作：** ざすこ（道草 雑草子）  
X: [@zasuko_michiksa](https://x.com/zasuko_michiksa) ／ YouTube: [ＡＩみちくさｃｈ](https://www.youtube.com/channel/UC84fyKjiilxssZVxhE_RiaA)


# ⚒️ Stable Diffusion WebUI Forge の起動
##＜はじめる前の準備＞※重要※
### ❶生成に必要な各種モデルファイルのDL用URLを格納する各フォルダの記入欄に記載して下さい。
└🌐GoogleColab上の 📂stable-diffusion-webui-forge / models  以下に、生成に必要な各種ファイルをDLします。
### ❷（▶）ボタンをクリックしてForgeを起動


In [ ]:
# @markdown # 👈この（▶）ボタンをクリックしてForgeを起動（☕起動まで数分かかります）
# @markdown ### ※停止と再起動もこのボタンで行います。
# @markdown ## 🌱実行ログに「Running on public URL: https://...」または「This is the URL to access...」が表示されたらクリックして開いてください。

# ==========================================
# 🔧 環境設定 (Python 3.10 via UV)
# ==========================================
import subprocess
import os
import sys
import time
import threading
import socket
from pathlib import Path
from google.colab import drive
from IPython.display import clear_output

print("🔧 Installing uv...")
!pip install -q uv

print("🐍 Creating Python 3.10 environment...")
venv_dir = "/content/venv"
if not os.path.exists(venv_dir):
    !uv venv {venv_dir} --python 3.10
else:
    print("🐍 Python 3.10 environment already exists. Skipping creation.")
venv_python = f"{venv_dir}/bin/python"

# ==========================================
# 📁 Google Drive マウント & 出力設定
# ==========================================
use_google_drive = True  # @param {type:"boolean"}

if use_google_drive:
    drive.mount('/content/drive')
    print("✅ Google Driveがマウントされました")
else:
    print("⏸️  Google Driveは使用しません")

# ==========================================
# 🛠️ Forgeのインストール (Haoming02 Classic)
# ==========================================
%cd /content
repo_url = "https://github.com/Haoming02/sd-webui-forge-classic"
repo_dir = "/content/stable-diffusion-webui-forge"
branch = "classic"

if os.path.exists(repo_dir):
    %cd {repo_dir}
    print("🔄 Updating existing repository...")
    !git remote set-url origin {repo_url}
    !git fetch origin
    !git checkout {branch}
    !git reset --hard origin/{branch}
else:
    print("🚀 Forgeリポジトリをクローン中 (Haoming02 Classic)...")
    !git clone {repo_url} {repo_dir} --branch {branch}
    %cd {repo_dir}
    !git checkout be2a221478896ea1873f1fdd4664de7112513571  # 2026-02-19 固定

# ==========================================
# 📦 依存関係のインストール (Venv)
# ==========================================
print("📦 Installing dependencies into Python 3.10 venv...")
# Downgrade setuptools for CLIP compatibility
!uv pip install --python {venv_dir} pip "setuptools<70" wheel

# Install Torch / xformers
!uv pip install --python {venv_dir} xformers!=0.0.18 torch==2.5.1 torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

# Install requirements.txt
if os.path.exists("requirements.txt"):
    !uv pip install --python {venv_dir} -r requirements.txt
else:
    print("⚠️ requirements.txt not found, skipping...")

# Pre-install CLIP with no build isolation (uses venv's setuptools<70)
print("🔧 Pre-installing CLIP...")
!uv pip install --python {venv_dir} --no-build-isolation "git+https://github.com/openai/CLIP.git@d50d76daa670286dd6cacf3bcd80b5e4823fc8e1"

# ==========================================
# 🩹 NumPy固定（Forge/Gradioの bool8 エラー対策）
# ==========================================
print("🩹 Checking NumPy version in venv...")

import subprocess

def ensure_numpy_1264(py_path):
    try:
        current_ver = subprocess.check_output(
            [py_path, "-c", "import numpy; print(numpy.__version__)"],
            text=True
        ).strip()
    except Exception:
        current_ver = None

    print(f"Current NumPy: {current_ver}")

    if current_ver != "1.26.4":
        print("🔧 Reinstalling NumPy 1.26.4 ...")
        subprocess.run([py_path, "-m", "pip", "uninstall", "-y", "numpy"], check=False)
        subprocess.run([py_path, "-m", "pip", "install", "numpy==1.26.4"], check=True)

        current_ver = subprocess.check_output(
            [py_path, "-c", "import numpy; print(numpy.__version__)"],
            text=True
        ).strip()

    print(f"✅ NumPy pinned to: {current_ver}")

ensure_numpy_1264(venv_python)

# ==========================================
# 📤 画像保存先の設定
# ==========================================
# @markdown ### 生成画像の保存先（Google Drive）
# @markdown 指定したフォルダが出力先としてリンクされます
# @markdown ### 生成画像の保存先（Google Drive）
# @markdown 指定したフォルダが出力先としてリンクされます
drive_output_path = "/content/drive/MyDrive/Forge_output"  # @param {type:"string"}

if use_google_drive and drive_output_path:
    os.makedirs(drive_output_path, exist_ok=True)

    # Fix for both 'outputs' and 'output' (singular/plural)
    # Also suppress xdg-open errors by creating a dummy script

    # 1. Suppress xdg-open errors (browser open attempts)
    !echo "#!/bin/bash" > /usr/bin/xdg-open
    !echo "exit 0" >> /usr/bin/xdg-open
    !chmod +x /usr/bin/xdg-open

    # 2. Link Output Directories
    target_dirs = ["outputs", "output"]
    for dir_name in target_dirs:
        full_path = f"/content/stable-diffusion-webui-forge/{dir_name}"

        # Remove existing if it's a directory (not a link) or incorrect link
        if os.path.exists(full_path):
            if not os.path.islink(full_path):
                print(f"⚠️ Removing directory to replace with link: {full_path}")
                !rm -rf {full_path}

        if not os.path.exists(full_path):
           !ln -s "{drive_output_path}" {full_path}
           print(f"✅ Linked {dir_name} to Google Drive: {drive_output_path}")
        else:
           print(f"ℹ️ {dir_name} is already linked/exists.")

# ==========================================
# 📁 Forge の一時保存フォルダ作成
# ==========================================
tmp_dir = "/content/stable-diffusion-webui-forge/tmp"
os.makedirs(tmp_dir, exist_ok=True)
print(f"✅ tmp folder ready: {tmp_dir}")

# ==========================================
# 🎁 必須パッケージ導入 & トンネル設定
# ==========================================
print("📦 Installing system dependencies...")
!apt-get -y install -qq aria2

# Cloudflaredのインストール
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!rm cloudflared-linux-amd64.deb
clear_output()

# ========================================
# 📥 モデルダウンロード関数定義
# ========================================
def download_with_aria2c(link, folder):
    filename = link.split("/")[-1].split('?')[0]
    command = [
        "aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M",
        "--connect-timeout=5", "--timeout=5", "--max-tries=1",
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
        link, "-d", folder, "-o", filename
    ]
    print(f"Downloading {filename}...")
    try:
        subprocess.run(command, shell=False, check=True)
        print(f"✅ Downloaded: {filename}")
    except subprocess.CalledProcessError:
        print(f"❌ Failed to download: {link}")
    return filename


import os
import re
import subprocess
import urllib.parse

import re

def get_filename_from_headers(url, token=None):
    # Use curl -I to fetch headers
    # We need to capture stdout
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
    cmd = f'curl -s -I -L -H "User-Agent: {user_agent}" "{url}"'

    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        headers = result.stdout

        # Look for filename in Content-Disposition
        # Content-Disposition: attachment; filename="Waifu_Diffusion.safetensors"
        match = re.search(r'filename="?([^";\n]+)"?', headers)
        if match:
            return match.group(1)

        # Fallback: filename* (UTF-8)
        match_utf = re.search(r"filename\*=UTF-8''([^;\n]+)", headers)
        if match_utf:
            return urllib.parse.unquote(match_utf.group(1))

    except Exception as e:
        print(f"Warning: Could not fetch headers: {e}")

    return None

def get_civitai_filename_via_api(model_version_id):
    import requests
    try:
        api_url = f"https://civitai.com/api/v1/model-versions/{model_version_id}"
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"}
        resp = requests.get(api_url, headers=headers, timeout=5)
        if resp.status_code == 200:
            data = resp.json()
            if 'files' in data and data['files']:
                for f in data['files']:
                    if f.get('type') == 'Model' and f.get('primary', False):
                        return f['name']
                return data['files'][0]['name']
    except Exception as e:
        print(f"   ⚠️ API Name Lookup Failed: {e}")
    return None

def download_civitai_model(civitai_link, civitai_token, folder):
    import requests
    import os
    import re
    from urllib.parse import unquote
    import hashlib
    from concurrent.futures import ThreadPoolExecutor
    from tqdm import tqdm
    import time

    os.makedirs(folder, exist_ok=True)

    # --- 1. Preparation ---
    url = civitai_link.strip()

    # Extract ID
    model_id = None
    match = re.search(r'/models/(\d+)', url)
    if match:
        model_id = match.group(1)

    # Append Token
    if civitai_token and "token=" not in url:
        sep = "&" if "?" in url else "?"
        url += f"{sep}token={civitai_token}"

    print(f"Downloading from Civitai...")
    safe_url = re.sub(r'token=[^&]+', 'token=******', url)

    # Headers (Mimic Chrome)
    HEADERS = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
        "Accept": "*/*",
        "Connection": "keep-alive"
    }

    # --- 2. Filename Resolution ---
    filename = None

    # A. API
    if model_id:
        # Re-using the helper function if available (get_civitai_filename_via_api)
        # We assume it is defined in the notebook scope
        try:
            filename = get_civitai_filename_via_api(model_id)
            if filename:
                print(f"   ✅ Found Name via API: {filename}")
        except NameError:
            pass # Helper not defined?

    # B. HEAD Request (Also needed for size/redirects)
    session = requests.Session()
    session.headers.update(HEADERS)

    final_url = url
    file_size = 0

    print("   🌐 Resolving URL and Size...")
    try:
        with session.head(url, allow_redirects=True, timeout=10) as h:
            final_url = h.url

            # Try to get filename from headers if still needed
            if not filename:
                cd = h.headers.get('Content-Disposition')
                if cd:
                    fn_match = re.search(r'filename="?([^";]+)"?', cd)
                    if fn_match:
                        filename = fn_match.group(1)
                    else:
                        fn_match_star = re.search(r"filename\*=UTF-8''([^;]+)", cd)
                        if fn_match_star:
                            filename = unquote(fn_match_star.group(1))

            if 'Content-Length' in h.headers:
                file_size = int(h.headers['Content-Length'])
    except Exception as e:
        print(f"   ⚠️ Connection Error during resolve: {e}")

    # C. ID Fallback
    if not filename:
         if model_id:
            filename = f"{model_id}.safetensors"
         else:
            filename = f"model.safetensors"

    filename = re.sub(r'[\\/*?:"<>|]', "", filename)
    filepath = os.path.join(folder, filename)
    print(f"   🎯 Target Filename: {filename}")

    if os.path.exists(filepath):
        if os.path.getsize(filepath) > 1024:
            print(f"   ✅ File already exists. Skipping.")
            return

    # --- 3. Custom Parallel Download ---

    if file_size < 10 * 1024 * 1024:
        # Small file: Single thread
        print(f"   📄 Small file ({file_size/1024/1024:.2f}MB). Downloading normally...")
        try:
            with session.get(final_url, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(filepath, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            print("   ✅ Download completed.")
        except Exception as e:
            print(f"   ❌ Download failed: {e}")
        return

    # Large file: Multi-thread
    num_threads = 4
    chunk_size = file_size // num_threads

    print(f"   🚀 Starting Custom Parallel Download ({num_threads} threads) of {file_size/1024/1024:.2f}MB...")

    # Prepare File
    with open(filepath, "wb") as f:
        f.seek(file_size - 1)
        f.write(b"\0")

    pbar = tqdm(total=file_size, unit='B', unit_scale=True, unit_divisor=1024, desc=filename, ncols=100)

    def download_chunk(idx):
        start = idx * chunk_size
        end = start + chunk_size - 1
        if idx == num_threads - 1:
            end = file_size - 1

        headers = HEADERS.copy()
        headers["Range"] = f"bytes={start}-{end}"

        try:
            # New session per thread is safer
            with requests.Session() as s:
                s.headers.update(headers)
                with s.get(final_url, stream=True, timeout=60) as r:
                    r.raise_for_status()

                    # We need to write to specific offset.
                    # File input/output with threads requires care.
                    # Best: write to memory or separate temp files?
                    # Memory is risky on Colab (RAM).
                    # 'r+b' on the main file with lock?
                    # os.pwrite is atomic-ish?
                    # Let's use open(filepath, 'r+b') inside thread?

                    with open(filepath, 'r+b') as f:
                        f.seek(start)
                        for chunk in r.iter_content(chunk_size=1024*1024):
                            if chunk:
                                f.write(chunk)
                                pbar.update(len(chunk))
        except Exception as e:
            print(f"   ⚠️ Thread {idx} failed: {e}")
            raise e

    try:
        with ThreadPoolExecutor(max_workers=num_threads) as executor:
            futures = [executor.submit(download_chunk, i) for i in range(num_threads)]
            for future in futures:
                future.result() # Wait and raise exceptions
        pbar.close()
        print("   ✅ Download completed.")
    except Exception as e:
        pbar.close()
        print(f"   ❌ Parallel download failed: {e}")
        print("   ⚠️ Falling back to sequential requests...")
        # Fallback to single thread requests (robust)
        try:
             with session.get(final_url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(filepath, 'wb') as f, tqdm(total=file_size, unit='B', unit_scale=True, ncols=100) as bar:
                    for chunk in r.iter_content(chunk_size=1024*1024):
                        f.write(chunk)
                        bar.update(len(chunk))
             print("   ✅ Download completed (Fallback).")
        except Exception as e2:
             print(f"   ❌ Fallback failed: {e2}")


def verify_file_integrity(folder, min_size_mb=10):
    if not os.path.exists(folder): return
    print(f"🔍 Verifying files in {folder}...")
    files = os.listdir(folder)
    if not files:
        print("   (Empty folder)")
        return

    for filename in files:
        filepath = os.path.join(folder, filename)

        # 1. Sanitize Filename (Fix ?Authorization=... AND Ensure Extension)
        new_filename = filename
        renamed = False

        # Strip query string
        if "?" in new_filename:
            new_filename = new_filename.split("?")[0]
            renamed = True

        # Check for extension
        name, ext = os.path.splitext(new_filename)
        if not ext or len(ext) < 2: # No extension or dot only
            new_filename = f"{new_filename}.safetensors"
            renamed = True

        if renamed and new_filename != filename:
            clean_path = os.path.join(folder, new_filename)
            try:
                os.rename(filepath, clean_path)
                print(f"   ✏️ Renamed: {filename} -> {new_filename}")
                filename = new_filename # Update var for next checks
                filepath = clean_path
            except Exception as e:
                print(f"   ⚠️ Failed to rename {filename}: {e}")

        # 2. Size Check
        if os.path.isfile(filepath):
            # Skip txt/json/yaml checks (config files)
            if filename.lower().endswith(('.txt', '.json', '.yaml', '.html')):
                continue

            size_mb = os.path.getsize(filepath) / (1024 * 1024)
            print(f"   📄 {filename}: {size_mb:.2f} MB")

            if size_mb < min_size_mb:
                print(f"   ❌ Too small (<{min_size_mb}MB). Identifying as corrupt HTML/Error page. Deleting...")
                try:
                    os.remove(filepath)
                    print("      Deleted.")
                except Exception as e:
                    print(f"      Failed to delete: {e}")
            else:
                print("      ✅ Valid.")




def process_urls(url_list_str, folder, token=None, min_size_mb=10):
    urls = [line.strip() for line in url_list_str.split('\n') if line.strip() and not line.startswith('#')]
    for url in urls:
        if not url: continue

        if "civitai.com" in url:
            download_civitai_model(url, token, folder)
        else:
            download_with_aria2c(url, folder)

    verify_file_integrity(folder, min_size_mb)

# @markdown ## 🔑 Civitai API Token
# @markdown Google Colab のシークレットに `CIVITAI_KEY` という名前で登録したAPIキーを自動で読み込みます。
from google.colab import userdata
try:
    civitai_token = userdata.get("CIVITAI_KEY")
    if not civitai_token:
        raise ValueError("empty")
    print("✅ Civitai API キーをシークレットから読み込みました")
except Exception:
    raise RuntimeError(
        "\n\n❌ Civitai API キーが設定されていません。\n"
        "左サイドバーの 🔑 シークレット に CIVITAI_KEY を登録してから再実行してください。\n"
        "取得URL: https://civitai.com/user/account"
    )

# @markdown ## 📁 Checkpoints (Main Models)
# @markdown 保存先: models/Stable-diffusion
checkpoint_url_1 = "https://civitai.com/api/download/models/2514310?type=Model&format=SafeTensor&size=pruned&fp=fp16"  # @param {type:"string"}
checkpoint_url_2 = ""  # @param {type:"string"}
checkpoint_url_3 = ""  # @param {type:"string"}
checkpoint_url_4 = ""  # @param {type:"string"}

checkpoints_urls = "\n".join([x for x in [checkpoint_url_1, checkpoint_url_2, checkpoint_url_3, checkpoint_url_4] if x.strip()])

# @markdown ## 📁 LoRAs
# @markdown 保存先: models/Lora
lora_url_1 = ""  # @param {type:"string"}
lora_url_2 = ""  # @param {type:"string"}
lora_url_3 = ""  # @param {type:"string"}
lora_url_4 = ""  # @param {type:"string"}

loras_urls = "\n".join([x for x in [lora_url_1, lora_url_2, lora_url_3, lora_url_4] if x.strip()])

# @markdown ## 📁 VAE
# @markdown 保存先: models/VAE
vae_urls = "https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors"  # @param {type:"string"}

# @markdown ## 📁 Embeddings (Textual Inversion)
# @markdown 保存先: embeddings
embedding_url_1 = ""  # @param {type:"string"}
embedding_url_2 = ""  # @param {type:"string"}
embedding_url_3 = ""  # @param {type:"string"}
embedding_url_4 = ""  # @param {type:"string"}

embeddings_urls = "\n".join([x for x in [embedding_url_1, embedding_url_2, embedding_url_3, embedding_url_4] if x.strip()])

# @markdown ## 📁 ControlNet Models
# @markdown 保存先: models/ControlNet
controlnet_url_1 = "https://huggingface.co/xinsir/controlnet-union-sdxl-1.0/resolve/main/diffusion_pytorch_model_promax.safetensors?download=true"  # @param {type:"string"}
controlnet_url_2 = "https://civitai.com/api/download/models/1297043?type=Model&format=SafeTensor&size=full&fp=fp16"  # @param {type:"string"}
controlnet_url_3 = ""  # @param {type:"string"}
controlnet_url_4 = ""  # @param {type:"string"}

controlnet_urls = "\n".join([x for x in [controlnet_url_1, controlnet_url_2, controlnet_url_3, controlnet_url_4] if x.strip()])

# ダウンロード実行
process_urls(checkpoints_urls, "/content/stable-diffusion-webui-forge/models/Stable-diffusion", civitai_token)
process_urls(loras_urls, "/content/stable-diffusion-webui-forge/models/Lora", civitai_token)
process_urls(vae_urls, "/content/stable-diffusion-webui-forge/models/VAE", civitai_token)
process_urls(embeddings_urls, "/content/stable-diffusion-webui-forge/embeddings", civitai_token, min_size_mb=0.001)
process_urls(controlnet_urls, "/content/stable-diffusion-webui-forge/models/ControlNet", civitai_token)
print("✅ ダウンロード処理完了")

# ========================================
# 🧩 拡張機能のインストール
# ========================================
# @markdown ## 🧩 Extensions (Optional)
# @markdown 起動時に自動インストールする拡張機能を選択してください
install_prompt_all_in_one = True  # @param {type:"boolean"}
install_adetailer = True  # @param {type:"boolean"}
install_config_presets = True  # @param {type:"boolean"}

extensions_dir = "/content/stable-diffusion-webui-forge/extensions"
os.makedirs(extensions_dir, exist_ok=True)

if install_prompt_all_in_one:
    repo_path = f"{extensions_dir}/sd-webui-prompt-all-in-one"
    if not os.path.exists(repo_path):
        print("🧩 Installing sd-webui-prompt-all-in-one...")
        !git clone https://github.com/Physton/sd-webui-prompt-all-in-one.git {repo_path}
        !git -C {repo_path} checkout 596a931725209a9ee271e6a2bd6d95a1cdb33778  # 2026-02-19 固定
    else:
        print("🧩 sd-webui-prompt-all-in-one already installed.")
    group_tags_dir = f"{repo_path}/group_tags"
    os.makedirs(group_tags_dir, exist_ok=True)
    !wget -q -O {group_tags_dir}/custom.yaml https://raw.githubusercontent.com/zasuko/forge-settings/main/Config-Presets/custom.yaml

if install_adetailer:
    repo_path = f"{extensions_dir}/adetailer"
    if not os.path.exists(repo_path):
        print("🧩 Installing adetailer...")
        !git clone https://github.com/Bing-su/adetailer.git {repo_path}
        !git -C {repo_path} checkout 3a599f5d4607d8f9d8b9fc5a15526197418dae1a  # 2026-02-19 固定
    else:
        print("🧩 adetailer already installed.")

if install_config_presets:
    repo_path = f"{extensions_dir}/Config-Presets"
    if not os.path.exists(repo_path):
        print("🧩 Installing Config-Presets...")
        !git clone https://github.com/Zyin055/Config-Presets.git {repo_path}
        !git -C {repo_path} checkout e252e60288b1104a45ea12106d92a8d145364fa1  # 2026-02-19 固定
    else:
        print("🧩 Config-Presets already installed.")
    !wget -q -O {repo_path}/config-txt2img.json https://raw.githubusercontent.com/zasuko/forge-settings/main/Config-Presets/config-txt2img.json
    !wget -q -O {repo_path}/config-img2img.json https://raw.githubusercontent.com/zasuko/forge-settings/main/Config-Presets/config-img2img.json

# ========================================
# 🚀 Forge 起動 & 接続監視
# ========================================
def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\n🚀 Forge started! Setting up tunnel...\n")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            cf_url = l[l.find("http"):].strip()
            print("\n" + "="*60)
            print(f"🔥 Cloudflare URL: {cf_url}")
            print("="*60 + "\n")

threading.Thread(target=iframe_thread, daemon=True, args=(7860,)).start()

print("🚀 Forgeを起動します...初期セットアップには数分かかる場合があります。")

# 起動引数 (REMOVED --uv to force standard python usage)
use_share_link = False  # @param {type:"boolean"}

args = [
    "--theme dark",
    "--enable-insecure-extension-access",
    "--cuda-malloc",
    "--cuda-stream",
    "--xformers"
]
if use_share_link:
    args.append("--share")

args_str = " ".join(args)

# Unset MPLBACKEND to avoid Colab-specific matplotlib crash in venv
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']

print(f"🚀 Launching with Python: {venv_python}")

# 起動直前にも NumPy を念押し固定
ensure_numpy_1264(venv_python)

# Explicitly use the venv python executable
!{venv_python} launch.py {args_str}


# 📝 メモ・リンク集
## おすすめモデルなど
- [Civitai](https://civitai.com/)
- [Hugging Face](https://huggingface.co/)
- [Stable Diffusion WebUI Forge - Classic (GitHub)](https://github.com/Haoming02/sd-webui-forge-classic)
